# Scale extension — does convergence strengthen with encoder capacity?

**Companion to `convergence_figures.ipynb`. This one is an EXPERIMENT, not a figure script.**

## What this tests, and why

Huh et al.'s Platonic Representation Hypothesis plots alignment against language-model capability up to 70B parameters and finds it rises. This project measured convergence at a much smaller scale — text encoders at bge-m3/BERT/GPT-2 size (768–1024-d, ~0.1–0.6B params) — which is the single most likely objection at the defense (see `Defense_Brief` §8, `Mock_Viva` Tier 2, `Project_Atlas` "Scale the text side").

This notebook tests the prediction directly, on **both** sides:

- **Image side:** add **DINOv2-giant** (1.1B params) to the existing small→base→large ladder. The project already measured +15.3 points of ceiling per decade of parameters (R² 0.949) across three rungs. A fourth rung tests whether that line holds or bends.
- **Text side:** add one or two larger text encoders. This is the axis PRH actually plots and the one the project never varied.

Two quantities are measured for each new encoder:

1. **Raw shape agreement (ρ)** — Spearman of pairwise-distance structure against every existing encoder. No map fitted, nothing trained. This is the PRH-style alignment measure.
2. **Hub transfer (% of native)** — fit one entry map into the **existing frozen hub**, apply the **existing frozen caption head**, score R@1 against a natively-fitted head. This is the project's own measure, and it is genuinely zero-shot for the head.

## Why both, and not just one

Section E.12 established that these two do **not** track each other: ConvNeXt has the lowest raw agreement of any image encoder (ρ = 0.330) yet the highest transfer (96.7%). So "does scale help?" has two possible answers, and they can differ. Measuring only one would be the same mistake E.6 made on the text side.

## Cost, honestly

- DINOv2-giant over 9,533 images: ~15–25 min on an L4.
- A 4B text encoder over 47,665 captions (5 per image, averaged as in the original protocol): ~45–70 min.
- New caches: roughly 100–350 MB each. Check Drive space first.
- Everything downstream (maps, hub entry, transfer) is closed-form and takes seconds — consistent with the project's "one GPU pass, then a ruler" claim.

## Run order

Cells 1–2 configure and **pre-register**. Cell 3 loads existing artefacts. Cell 4 is the one you must edit — your COCO source. Cells 5–6 encode. Cells 7–9 measure. Cell 10 scores the result against the prediction written in Cell 2.

Stages cache to Drive and skip if already done, so the notebook is resumable.

In [ ]:
# Cell 1 — mount, install, configure
from google.colab import drive
drive.mount('/content/drive')

!pip -q install transformers timm sentence-transformers --upgrade 2>&1 | tail -1

import os, json, numpy as np, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

FOLDER   = "/content/drive/MyDrive/convergence_experiment"
OUT      = os.path.join(FOLDER, "scale_extension")
os.makedirs(OUT, exist_ok=True)
DEV      = "cuda" if torch.cuda.is_available() else "cpu"

# ── which models to add ──────────────────────────────────────────────
# Image side: one more rung on the existing DINOv2 ladder.
#   The project uses cls+patch concatenation, so output dim = 2 x native.
#   small 384->768, base 768->1536, large 1024->2048, giant 1536->3072.
IMAGE_MODELS = {
    "img_giant": dict(hf="facebook/dinov2-giant", params_b=1.14, native_dim=1536),
}

# Text side: a capacity ladder. Start with the smaller one; add the larger
# only if the first completes and Drive has room.
TEXT_MODELS = {
    "txt_qwen06": dict(hf="Qwen/Qwen3-Embedding-0.6B", params_b=0.6),
    # "txt_qwen4": dict(hf="Qwen/Qwen3-Embedding-4B", params_b=4.0),   # uncomment for the second rung
}

# Existing encoders' parameter counts, for the capacity-ladder plot.
# (approximate, in billions — used only for the x-axis)
KNOWN_PARAMS = {
    "img_small": 0.022, "img_base": 0.087, "img_large": 0.304, "img_giant": 1.14,
    "txt_bge": 0.568, "txt_bert": 0.110, "txt_gpt2": 0.124, "txt_sbert": 0.110,
    "convnext": 0.089,
}

BATCH_IMG, BATCH_TXT = 32, 64
print("output dir:", OUT)

In [ ]:
# Cell 2 — PRE-REGISTRATION. Runs before any measurement, writes to disk.
#
# The project's standing rule (report E.9): predictions are written down before
# the numbers exist. This cell is that record. Edit the predictions if you
# disagree with them — but edit them NOW, not after seeing results.

PREREG_PATH = os.path.join(OUT, "prereg_scale_extension.json")

PREREG = {
  "written_before_any_measurement": True,
  "hypothesis": (
    "PRH predicts alignment rises with encoder capacity. The project's own capacity "
    "ladder (+15.3 pts of ceiling per decade, R2 0.949, three rungs) predicts the same "
    "for hub transfer on the image side."
  ),
  "predictions": {
    "P1_image_transfer": (
      "DINOv2-giant transfers at or above DINOv2-large's 92.9 percent of native. "
      "CONFIRMED if >= 92.9. A LOWER value falsifies the naive reading of the capacity "
      "ladder and would be the more interesting result."
    ),
    "P2_image_rho": (
      "DINOv2-giant's raw shape agreement with the text encoders is >= DINOv2-large's "
      "best (0.418 with BERT). CONFIRMED if >= 0.418."
    ),
    "P3_text_rho": (
      "A larger text encoder shows HIGHER raw shape agreement with the image encoders "
      "than bge-m3's best cross-modal value (0.320 with DINOv2-large). This is the "
      "direct test of the scale caveat. CONFIRMED if >= 0.320."
    ),
    "P4_decoupling": (
      "Following E.12, rho and transfer need NOT move together. If one rises and the "
      "other does not, that REPLICATES E.12 rather than contradicting anything."
    ),
  },
  "threshold_note": (
    "The project treats a 2-point move in transfer as the noise floor (see G10). "
    "Differences below 2 points are not claimed as effects."
  ),
  "what_would_falsify": (
    "P1 falsified if giant transfers below 90.9 percent (more than 2 points under large). "
    "P3 falsified if the larger text encoder's best cross-modal rho is below 0.320."
  ),
}

if os.path.exists(PREREG_PATH):
    PREREG = json.load(open(PREREG_PATH))
    print("Pre-registration already on disk (not overwritten):")
else:
    json.dump(PREREG, open(PREREG_PATH, "w"), indent=2)
    print("Pre-registration WRITTEN to", PREREG_PATH)
for k, v in PREREG["predictions"].items():
    print(f"\n  {k}: {v}")

In [ ]:
# Cell 3 — load the existing frozen hub, head, and encoder caches
#
# Nothing here is refitted. The hub basis and the caption head were fitted once,
# on the 8,533 training rows, and are used read-only.

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

# raw vectors for the 7 hub encoders, row-aligned, 9,533 items
z = np.load(os.path.join(FOLDER, "hub_rebuilt.npz"), allow_pickle=True)
names = [str(x) for x in z["encoder_names"]]
raw   = {n: np.asarray(z[f"raw_{n}"]) for n in names}
N     = raw[names[0]].shape[0]
hub_ids = np.asarray(z["hub_ids"]) if "hub_ids" in z.files else np.arange(N)
print("existing encoders:", names, "| N =", N)

# the frozen 4-space hub + caption head (the PUBLISHED protocol)
hx = np.load(os.path.join(FOLDER, "hub_exact.npz"), allow_pickle=True)
print("\nhub_exact.npz keys:", list(hx.files)[:14], "...")
HUB_BASIS = np.asarray(hx["basis"])          # (5376, 512)
HUB_MU    = np.asarray(hx["mu"])
HUB_SV    = np.asarray(hx["sv"])
W_HEAD    = np.asarray(hx["head"])           # (512, 1024) hub -> bge
TRAIN_IDX = np.asarray(hx["train_idx"])      # 8,533
EVAL_IDX  = np.asarray(hx["eval_idx"])       # 1,000
ALPHA     = float(hx["alpha"]) if "alpha" in hx.files else 1e-2
print(f"\nhub basis {HUB_BASIS.shape} | head {W_HEAD.shape} | "
      f"train {len(TRAIN_IDX)} eval {len(EVAL_IDX)} | alpha {ALPHA}")

# hub coordinates for all items (needed as the regression target for new entry maps)
H_ALL = np.asarray(z["H_all"]) if "H_all" in z.files else None
if H_ALL is None:
    raise RuntimeError("H_all not found; needed as the entry-map target")
print("H_all:", H_ALL.shape)

# the bge caption targets (the head's output space, and the retrieval gallery)
T_BGE = np.asarray(z["raw_txt_bge"])
print("T_BGE:", T_BGE.shape)

# ConvNeXt, for context in the ladder plots
CONV_FILE = os.path.join(FOLDER, "e1_img_ckpt_convnext-base-224-22k_native.npz")
if os.path.exists(CONV_FILE):
    zc = np.load(CONV_FILE, allow_pickle=True)
    for k in zc.files:
        a = np.asarray(zc[k])
        if a.ndim == 2 and a.shape[0] == N and a.shape[1] >= 512:
            raw["convnext"] = a; names = names + ["convnext"]; break
    print("ConvNeXt added:", raw["convnext"].shape if "convnext" in raw else "not found")

LAB = {"img_small":"DINOv2-small","img_base":"DINOv2-base","img_large":"DINOv2-large",
       "img_giant":"DINOv2-giant","txt_bge":"bge-m3","txt_gpt2":"GPT-2","txt_bert":"BERT",
       "txt_sbert":"SBERT","convnext":"ConvNeXt-base",
       "txt_qwen06":"Qwen3-Emb-0.6B","txt_qwen4":"Qwen3-Emb-4B"}
def lab(n): return LAB.get(n, n)

In [ ]:
# Cell 4 — resolve the same COCO items the hub was built from
#
# This follows the download/cache pattern already used in B1_extract_pairs:
# fetch the plain zips from cocodataset.org, validate before trusting a cached
# file, extract under DATA_DIR/coco. No HF loading scripts.
#
# WHAT IS DIFFERENT FROM B1. B1 builds Experiment B: COCO val2017, 4,000 images,
# FIRST caption per image. The hub corpus is 9,533 items with FIVE captions
# AVERAGED - a different protocol from a different (E-series) notebook. Because
# val2017 holds only 5,000 images, the hub items cannot all come from val2017,
# so this cell AUTO-DETECTS which split actually contains your hub_ids rather
# than assuming one.
#
# ROW ALIGNMENT IS NON-NEGOTIABLE. Row i of every new cache must be the same
# COCO item as row i of every existing cache. Misalignment does not raise - it
# silently invalidates every rho and every transfer number. The asserts below
# are the guard.

import json as _json, urllib.request, zipfile
from collections import defaultdict
from pathlib import Path

COCO_ROOT = Path(os.environ.get("COCO_DIR", os.path.join(FOLDER, "coco")))
COCO_ROOT.mkdir(parents=True, exist_ok=True)
CAPTIONS_PER_ITEM = 5          # the hub protocol averages five captions

def _usable(z_path):
    """exists() alone is not enough - a truncated zip passes it and then
    fails confusingly during extraction."""
    if not z_path.exists() or z_path.stat().st_size < 1_000_000:
        return False
    try:
        with zipfile.ZipFile(z_path) as z:
            return z.testzip() is None
    except zipfile.BadZipFile:
        return False

def _fetch(url, dest_zip):
    if _usable(dest_zip):
        print(f"  using cached {dest_zip.name} ({dest_zip.stat().st_size/1e6:.0f} MB)")
    else:
        if dest_zip.exists():
            print(f"  {dest_zip.name} truncated or corrupt - re-downloading")
            dest_zip.unlink()
        print(f"  downloading {url} ...")
        tmp = dest_zip.with_suffix(".part")
        urllib.request.urlretrieve(url, str(tmp))
        tmp.rename(dest_zip)
        assert _usable(dest_zip), f"{dest_zip.name} unreadable after download"
    with zipfile.ZipFile(dest_zip) as z:
        z.extractall(COCO_ROOT)

# annotations cover both splits and are needed either way
if not (COCO_ROOT / "annotations" / "captions_val2017.json").exists():
    _fetch("http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
           COCO_ROOT / "ann.zip")

# --- auto-detect which split holds the hub items ---
want = set(int(i) for i in hub_ids)
chosen = None
for split, zurl in (("val2017",   "http://images.cocodataset.org/zips/val2017.zip"),
                    ("train2017", "http://images.cocodataset.org/zips/train2017.zip")):
    ann_path = COCO_ROOT / "annotations" / f"captions_{split}.json"
    if not ann_path.exists():
        continue
    ann = _json.load(open(ann_path))
    ids = set(im["id"] for im in ann["images"])
    covered = len(want & ids)
    print(f"  {split}: {covered}/{len(want)} hub items present")
    if covered == len(want):
        chosen = (split, ann, zurl); break

if chosen is None:
    raise RuntimeError(
        "No single COCO split contains all hub_ids. The hub corpus may be a "
        "union of splits or a different dataset entirely - check the E-series "
        "notebook that built e1_img_ckpt_*.npz and set COCO_DIR / the split "
        "manually before rerunning.")

SPLIT, ANN, ZURL = chosen
print("")
print(f"hub items resolve against COCO {SPLIT}")
if not (COCO_ROOT / SPLIT).exists():
    _fetch(ZURL, COCO_ROOT / f"{SPLIT}.zip")

# --- build aligned image paths and caption groups ---
id2file = {im["id"]: im["file_name"] for im in ANN["images"]}
caps = defaultdict(list)
for a in ANN["annotations"]:
    caps[a["image_id"]].append(a["caption"])

IMAGE_PATHS, CAPTIONS, short = [], [], 0
for i in hub_ids:
    i = int(i)
    IMAGE_PATHS.append(str(COCO_ROOT / SPLIT / id2file[i]))
    c = caps[i][:CAPTIONS_PER_ITEM]
    if len(c) < CAPTIONS_PER_ITEM:
        short += 1
        c = c + [c[-1]] * (CAPTIONS_PER_ITEM - len(c))   # pad by repeating the last
    CAPTIONS.append(c)

# --- alignment guards ---
assert len(IMAGE_PATHS) == N, f"{len(IMAGE_PATHS)} paths vs N={N}"
assert len(CAPTIONS)   == N, f"{len(CAPTIONS)} caption groups vs N={N}"
missing = [p for p in IMAGE_PATHS[:50] if not os.path.exists(p)]
assert not missing, f"image paths do not resolve, e.g. {missing[:2]}"

print("")
print(f"resolved {len(IMAGE_PATHS)} images and {len(CAPTIONS)} caption groups")
print(f"  first item: {os.path.basename(IMAGE_PATHS[0])}")
print(f"  its captions: {CAPTIONS[0][:2]} ...")
if short:
    print(f"  NOTE: {short} items had fewer than {CAPTIONS_PER_ITEM} captions (padded)")
print("")
print("ALIGNMENT OK - row i corresponds to hub_ids[i] throughout")

In [ ]:
# Cell 5 — encode the new IMAGE model(s). Resumable: skips if the cache exists.
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

@torch.no_grad()
def encode_images(hf_name, paths, batch=BATCH_IMG):
    """cls+patch concatenation, matching the project's existing image caches."""
    proc  = AutoImageProcessor.from_pretrained(hf_name)
    model = AutoModel.from_pretrained(hf_name, torch_dtype=torch.float16).to(DEV).eval()
    out = []
    for s in range(0, len(paths), batch):
        imgs = [Image.open(p).convert("RGB") for p in paths[s:s+batch]]
        px = proc(images=imgs, return_tensors="pt").to(DEV)
        h  = model(**px).last_hidden_state            # (B, 1+P, D)
        cls   = h[:, 0]                                # (B, D)
        patch = h[:, 1:].mean(1)                       # (B, D)
        out.append(torch.cat([cls, patch], -1).float().cpu().numpy())
        if (s // batch) % 20 == 0:
            print(f"    {s}/{len(paths)}", flush=True)
    del model; torch.cuda.empty_cache()
    return np.concatenate(out, 0)

for key, cfg in IMAGE_MODELS.items():
    path = os.path.join(OUT, f"cache_{key}.npz")
    if os.path.exists(path):
        raw[key] = np.load(path)["v"]
        if key not in names: names.append(key)
        print(f"[{key}] cached already: {raw[key].shape}")
        continue
    print(f"[{key}] encoding {cfg['hf']} ...")
    V = encode_images(cfg["hf"], IMAGE_PATHS)
    assert V.shape[0] == N, f"row count {V.shape[0]} != {N}"
    np.savez_compressed(path, v=V, hf=cfg["hf"], params_b=cfg["params_b"])
    raw[key] = V
    if key not in names: names.append(key)
    print(f"[{key}] done: {V.shape}  (expected {2*cfg['native_dim']}d)")

In [ ]:
# Cell 6 — encode the new TEXT model(s). Averages CAPTIONS_PER_ITEM per item,
# matching the project's protocol ("five captions embedded then averaged").
from transformers import AutoTokenizer, AutoModel as AutoTextModel

@torch.no_grad()
def encode_texts(hf_name, caption_groups, batch=BATCH_TXT):
    tok   = AutoTokenizer.from_pretrained(hf_name)
    model = AutoTextModel.from_pretrained(hf_name, torch_dtype=torch.float16).to(DEV).eval()
    flat  = [c for grp in caption_groups for c in grp]     # N * CAPTIONS_PER_ITEM
    k     = len(caption_groups[0])
    vecs  = []
    for s in range(0, len(flat), batch):
        chunk = flat[s:s+batch]
        enc = tok(chunk, padding=True, truncation=True, max_length=64,
                  return_tensors="pt").to(DEV)
        h = model(**enc).last_hidden_state                  # (B, L, D)
        mask = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
        pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)   # mean-pool
        vecs.append(pooled.float().cpu().numpy())
        if (s // batch) % 50 == 0:
            print(f"    {s}/{len(flat)}", flush=True)
    del model; torch.cuda.empty_cache()
    V = np.concatenate(vecs, 0).reshape(len(caption_groups), k, -1)
    return V.mean(1)                                        # average the k captions

for key, cfg in TEXT_MODELS.items():
    path = os.path.join(OUT, f"cache_{key}.npz")
    if os.path.exists(path):
        raw[key] = np.load(path)["v"]
        if key not in names: names.append(key)
        print(f"[{key}] cached already: {raw[key].shape}")
        continue
    print(f"[{key}] encoding {cfg['hf']} ...")
    V = encode_texts(cfg["hf"], CAPTIONS)
    assert V.shape[0] == N, f"row count {V.shape[0]} != {N}"
    np.savez_compressed(path, v=V, hf=cfg["hf"], params_b=cfg["params_b"])
    raw[key] = V
    if key not in names: names.append(key)
    print(f"[{key}] done: {V.shape}")

print("\nall encoders now available:", names)

In [ ]:
# Cell 7 — MEASUREMENT 1: raw shape agreement (rho) for the new encoders
#
# Same method as the main figures notebook: Spearman of pairwise cosine distances
# over all N items, no map fitted. Memory-safe unit-normalised float32 ranks —
# raw 45.4M ranks would exceed float32's 16.7M exact-integer limit, so ranks are
# centred and unit-normalised in float64 first, making each Spearman a dot product.
from scipy.spatial.distance import pdist
from scipy.stats import rankdata

def rank_vec(X):
    d = pdist(l2n(X), metric="cosine")
    r = rankdata(d).astype(np.float64); del d
    r -= r.mean(); r /= np.linalg.norm(r)
    return r.astype(np.float32)

ranks = {}
for n_ in names:
    ranks[n_] = rank_vec(raw[n_])
    print("    ranked", lab(n_))

K = len(names); RHO = np.eye(K)
for i in range(K):
    for j in range(i+1, K):
        RHO[i,j] = RHO[j,i] = float(np.dot(ranks[names[i]], ranks[names[j]]))
del ranks
np.savez(os.path.join(OUT, "rho_matrix_extended.npz"), names=names, M=RHO, n_items=N)

IMG = {"img_small","img_base","img_large","img_giant","convnext"}
def best_cross(n_):
    """best rho against the opposite modality"""
    i = names.index(n_); same_img = n_ in IMG
    cands = [(RHO[i,j], names[j]) for j in range(K)
             if j != i and ((names[j] in IMG) != same_img)]
    return max(cands) if cands else (float("nan"), None)

print("\n=== raw shape agreement, new encoders ===")
for key in list(IMAGE_MODELS) + list(TEXT_MODELS):
    if key not in names: continue
    i = names.index(key)
    v, partner = best_cross(key)
    print(f"\n  {lab(key)}:")
    print(f"    best cross-modal rho = {v:.3f}  (with {lab(partner)})")
    for j in range(K):
        if j != i:
            print(f"      vs {lab(names[j]):16s} {RHO[i,j]:.3f}")

print("\n=== reference points from the existing set ===")
for n_ in ["img_large","txt_bge","convnext"]:
    if n_ in names:
        v, p_ = best_cross(n_)
        print(f"  {lab(n_):16s} best cross-modal rho = {v:.3f} (with {lab(p_)})")

In [ ]:
# Cell 8 — MEASUREMENT 2: hub transfer through the FROZEN hub and FROZEN head
#
# Exactly the ConvNeXt protocol (report E.2 / the frozen-hub document):
#   1. fit ONE entry map  X_new -> H   on the 8,533 training rows  (closed form)
#   2. apply the EXISTING frozen head  H -> bge   unchanged        (zero-shot)
#   3. score R@1 on the 1,000 held-out rows against a natively-fitted head
# The hub is not rebuilt and the head is not refitted. Nothing upstream moves.

def ridge(A, B, alpha=ALPHA):
    return np.linalg.solve(A.T @ A + alpha*np.eye(A.shape[1]), A.T @ B)

def r_at_1(P, G):
    sims = l2n(P) @ l2n(G).T
    return float((np.argmax(sims, 1) == np.arange(len(P))).mean())

GAL = T_BGE[EVAL_IDX]           # held-out bge gallery
transfer = {}

for key in list(IMAGE_MODELS) + list(TEXT_MODELS) + ["img_large"]:   # img_large as the control
    if key not in raw: continue
    X = np.asarray(raw[key], dtype=np.float64)

    # 1. entry map into the frozen hub, fitted on TRAIN rows only
    W_entry = ridge(X[TRAIN_IDX], H_ALL[TRAIN_IDX])
    H_new   = X @ W_entry

    # 2. frozen head, unchanged -> predictions on held-out rows
    pred_transfer = H_new[EVAL_IDX] @ W_HEAD

    # 3. native reference: a head fitted for THIS encoder specifically
    W_native = ridge(H_new[TRAIN_IDX], T_BGE[TRAIN_IDX])
    pred_native = H_new[EVAL_IDX] @ W_native

    # 4. random-map control
    rng = np.random.default_rng(0)
    W_rand = rng.standard_normal(W_entry.shape) * (W_entry.std())
    pred_rand = (X @ W_rand)[EVAL_IDX] @ W_HEAD

    r_t, r_n, r_c = r_at_1(pred_transfer, GAL), r_at_1(pred_native, GAL), r_at_1(pred_rand, GAL)
    pct = 100.0 * r_t / max(r_n, 1e-9)
    transfer[key] = dict(r1_transfer=r_t, r1_native=r_n, r1_control=r_c, pct_of_native=pct)
    print(f"{lab(key):18s} R@1 transfer {r_t:.3f} | native {r_n:.3f} | "
          f"{pct:5.1f}% of native | control {r_c:.4f}")

json.dump({k: {kk: float(vv) for kk, vv in v.items()} for k, v in transfer.items()},
          open(os.path.join(OUT, "transfer_extended.json"), "w"), indent=2)
print("\nReference: DINOv2-base 96.5, DINOv2-large 92.9, SigLIP 94.2, ConvNeXt 96.7 (chance 0.001)")

In [ ]:
# Cell 9 — the capacity ladder, both measures against parameter count
import matplotlib.pyplot as plt

NAVY, BLUE, TEAL, RED, GREY = "#172B54", "#2B5FD9", "#0F766E", "#B02418", "#6B7280"
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":8.5,"text.color":NAVY,
                     "axes.labelcolor":NAVY,"xtick.color":NAVY,"ytick.color":NAVY,
                     "axes.titlesize":9.5,"axes.titleweight":"bold"})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.0, 4.2), dpi=200)

# left: raw cross-modal rho vs params
pts = [(KNOWN_PARAMS.get(n_, np.nan), best_cross(n_)[0], n_)
       for n_ in names if n_ in KNOWN_PARAMS]
img_pts = [(p,r,n_) for p,r,n_ in pts if n_ in IMG and not np.isnan(p)]
txt_pts = [(p,r,n_) for p,r,n_ in pts if n_ not in IMG and not np.isnan(p)]
for grp, col, mk, lbl in ((img_pts, BLUE, "o", "image encoders"),
                          (txt_pts, TEAL, "^", "text encoders")):
    grp = sorted(grp)
    ax1.plot([p for p,_,_ in grp], [r for _,r,_ in grp], mk+"-", color=col, ms=6, lw=1.4, label=lbl)
    for p, r, n_ in grp:
        ax1.annotate(lab(n_), (p, r), fontsize=6.5, xytext=(3,4), textcoords="offset points")
ax1.set_xscale("log"); ax1.set_xlabel("parameters (billions, log)")
ax1.set_ylabel("best cross-modal rho")
ax1.set_title("Raw shape agreement vs capacity\n(no map fitted)")
ax1.grid(alpha=0.25, lw=0.5); ax1.legend(frameon=False, fontsize=7.5)

# right: hub transfer vs params
tp = sorted([(KNOWN_PARAMS.get(k, np.nan), v["pct_of_native"], k)
             for k, v in transfer.items() if k in KNOWN_PARAMS])
ax2.plot([p for p,_,_ in tp], [v for _,v,_ in tp], "o-", color=BLUE, ms=6, lw=1.4)
for p, v, k in tp:
    ax2.annotate(f"{lab(k)}\n{v:.1f}%", (p, v), fontsize=6.5, xytext=(3,-10),
                 textcoords="offset points")
ax2.axhspan(93.8, 96.5, color=TEAL, alpha=0.12)
ax2.text(ax2.get_xlim()[0]*1.2, 95.2, "within-family band 93.8-96.5", fontsize=6.8, color=TEAL)
ax2.set_xscale("log"); ax2.set_xlabel("parameters (billions, log)")
ax2.set_ylabel("% of natively-fitted head")
ax2.set_title("Hub transfer vs capacity\n(frozen hub, frozen head)")
ax2.grid(alpha=0.25, lw=0.5)

fig.suptitle("Does convergence strengthen with encoder capacity?", fontsize=11.5,
             fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUT, "scale_ladder.png"), bbox_inches="tight")
plt.show(); plt.close(fig)
print("saved scale_ladder.png")

In [ ]:
# Cell 10 — score the results against the pre-registration from Cell 2
#
# The predictions were written before any number existed. This cell reports
# CONFIRMED / FALSIFIED against them, either way, and records the verdict to disk.

pre = json.load(open(PREREG_PATH))
verdict = {}
print("=" * 72)
print("VERDICT AGAINST PRE-REGISTRATION")
print("=" * 72)

# P1 — image transfer
if "img_giant" in transfer:
    got = transfer["img_giant"]["pct_of_native"]
    ok  = got >= 92.9
    marg = got >= 90.9
    verdict["P1_image_transfer"] = dict(value=got, confirmed=bool(ok))
    print(f"\nP1  DINOv2-giant transfer >= 92.9% (DINOv2-large)")
    print(f"    measured {got:.1f}%  ->  {'CONFIRMED' if ok else ('WITHIN NOISE' if marg else 'FALSIFIED')}")
    if not ok:
        print("    NOTE: a lower value is the more interesting result - it bounds the")
        print("    capacity ladder rather than extending it. Report it as measured.")

# P2 — image rho
if "img_giant" in names:
    got, partner = best_cross("img_giant")
    ok = got >= 0.418
    verdict["P2_image_rho"] = dict(value=got, partner=partner, confirmed=bool(ok))
    print(f"\nP2  DINOv2-giant best cross-modal rho >= 0.418 (DINOv2-large)")
    print(f"    measured {got:.3f} with {lab(partner)}  ->  {'CONFIRMED' if ok else 'FALSIFIED'}")

# P3 — the scale caveat itself
for tk in TEXT_MODELS:
    if tk in names:
        got, partner = best_cross(tk)
        ok = got >= 0.320
        verdict[f"P3_text_rho_{tk}"] = dict(value=got, partner=partner, confirmed=bool(ok))
        print(f"\nP3  {lab(tk)} best cross-modal rho >= 0.320 (bge-m3's best)")
        print(f"    measured {got:.3f} with {lab(partner)}  ->  {'CONFIRMED' if ok else 'FALSIFIED'}")
        print("    THIS is the direct test of the scale caveat in Defense_Brief section 8.")

# P4 — decoupling (E.12 replication)
if "img_giant" in transfer and "img_giant" in names:
    d_rho  = best_cross("img_giant")[0] - 0.418
    d_pct  = transfer["img_giant"]["pct_of_native"] - 92.9
    same   = (d_rho >= 0) == (d_pct >= 0)
    verdict["P4_decoupling"] = dict(d_rho=d_rho, d_pct=d_pct, moved_together=bool(same))
    print(f"\nP4  do rho and transfer move together?")
    print(f"    d_rho {d_rho:+.3f}, d_transfer {d_pct:+.1f} pts  ->  "
          f"{'same direction' if same else 'OPPOSITE - replicates E.12'}")

json.dump({k: {kk: (float(vv) if isinstance(vv,(int,float,np.floating)) else vv)
               for kk, vv in v.items()} for k, v in verdict.items()},
          open(os.path.join(OUT, "verdict_scale_extension.json"), "w"), indent=2)

print("\n" + "=" * 72)
print("All artefacts in:", OUT)
print("  prereg_scale_extension.json   - written before measuring")
print("  rho_matrix_extended.npz       - full pairwise rho, all encoders")
print("  transfer_extended.json        - R@1 transfer / native / control")
print("  scale_ladder.png              - the two-panel capacity figure")
print("  verdict_scale_extension.json  - scored against the pre-registration")
print("\nSend the PNG and the two JSONs to Claude to fold into the documents.")